In [187]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [188]:
# Define the industries of GICS classification
gics_industries = {
    "101010": "Energy Equipment & Services",
    "101020": "Oil, Gas & Consumable Fuels",
    "151010": "Chemicals",
    "151020": "Construction Materials",
    "151030": "Containers & Packaging",
    "151040": "Metals & Mining",
    "151050": "Paper & Forest Products",
    "201010": "Aerospace & Defense",
    "201020": "Building Products",
    "201030": "Construction & Engineering",
    "201040": "Electrical Equipment",
    "201050": "Industrial Conglomerates",
    "201060": "Machinery",
    "201070": "Trading Companies & Distributors",
    "202010": "Commercial Services & Supplies",
    "202020": "Professional Services",
    "203010": "Air Freight & Logistics",
    "203020": "Passenger Airlines",
    "203030": "Marine Transportation",
    "203040": "Ground Transportation",
    "203050": "Transportation Infrastructure",
    "251010": "Automobile Components",
    "251020": "Automobiles",
    "252010": "Household Durables",
    "252020": "Leisure Products",
    "252030": "Textiles, Apparel & Luxury Goods",
    "253010": "Hotels, Restaurants & Leisure",
    "253020": "Diversified Consumer Services",
    "255010": "Distributors",
    "255030": "Broadline Retail",
    "255040": "Specialty Retail",
    "301010": "Consumer Staples Distribution & Retail",
    "302010": "Beverages",
    "302020": "Food Products",
    "302030": "Tobacco",
    "303010": "Household Products",
    "303020": "Personal Care Products",
    "351010": "Health Care Equipment & Supplies",
    "351020": "Health Care Providers & Services",
    "351030": "Health Care Technology",
    "352010": "Biotechnology",
    "352020": "Pharmaceuticals",
    "352030": "Life Sciences Tools & Services",
    "401010": "Banks",
    "402010": "Financial Services",
    "402020": "Consumer Finance",
    "402030": "Capital Markets",
    "402040": "Mortgage Real Estate Investment Trusts (REITs)",
    "403010": "Insurance",
    "451020": "IT Services",
    "451030": "Software",
    "452010": "Communications Equipment",
    "452020": "Technology Hardware, Storage & Peripherals",
    "452030": "Electronic Equipment, Instruments & Components",
    "453010": "Semiconductors & Semiconductor Equipment",
    "501010": "Diversified Telecommunication Services",
    "501020": "Wireless Telecommunication Services",
    "502010": "Media",
    "502020": "Entertainment",
    "502030": "Interactive Media & Services",
    "551010": "Electric Utilities",
    "551020": "Gas Utilities",
    "551030": "Multi-Utilities",
    "551040": "Water Utilities",
    "551050": "Independent Power and Renewable Electricity Producers",
    "601010": "Diversified REITs",
    "601025": "Industrial REITs",
    "601030": "Hotel & Resort REITs",
    "601040": "Office REITs",
    "601050": "Health Care REITs",
    "601060": "Residential REITs",
    "601070": "Retail REITs",
    "601080": "Specialized REITs",
    "602010": "Real Estate Management & Development",
    "na" : "Unclassified"
}

In [189]:
# Duplicate stocks in the S&P 500
duplicate_stocks = { 
    "50203010" : "GOOGL",
    "50202010" : "DISCK",
    "50201020" : "FOXA",
    "50201040" : "NWSA"
    }

In [190]:
# Get the industry of a company based on its GICS code
def get_industry(gics_code):
    
    if gics_code == np.nan:
        return "na"
    
    gics_code = str(gics_code)
    if gics_code[:6] in gics_industries:
        return gics_code[:6]
    else:
        return "na"

In [191]:
# Replace column names with actual gics codes
tickers = pd.read_csv('../Data/data_us/tickers.csv', header=None, dtype=str)
tickers.columns = ['ticker', 'gics_code']
tickers.set_index('ticker', inplace=True)

tickers['industry'] = tickers['gics_code'].apply(get_industry)

display(tickers)

,gics_code,industry
ticker,,
0111145D,NaN,na
0202445Q,NaN,na
0203524D,NaN,na
0226226D,NaN,na
0544749D,NaN,na
...,...,...
YUM,25301040,253010
ZBH,35101010,351010
ZBRA,45203010,452030


In [192]:
close_prices = pd.read_csv('../Data/data_us/adjusted.csv')

close_prices['Date'] = pd.to_datetime(close_prices['Date'], format = "%Y%m%d")
close_prices = close_prices.set_index('Date')

close_prices.fillna(method='ffill', inplace=True)
close_prices.fillna(method='bfill', inplace=True)

for key, value in duplicate_stocks.items():
    close_prices.drop(value, axis=1, inplace=True)

display(close_prices.head())

,0111145D,0202445Q,0203524D,0226226D,0544749D,0574018D,0772031D,0848680D,0867887D,0910150D,...,XOM,XRAY,XRX,XTO,XYL,YUM,ZBH,ZBRA,ZION,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2004-01-02,23.60,42.99,17.6344,46.1555,21.8167,23.1976,3.3329,14.1172,25.05,16.7114,...,20.4223,18.9882,19.3182,12.3424,20.298,8.2905,60.8751,43.5867,40.6011,23.82
2004-01-05,23.72,43.06,18.2824,45.3553,22.2326,23.2143,3.5783,14.6964,25.05,16.8684,...,20.8998,18.6737,19.6451,12.4431,20.298,8.4117,60.3272,43.6667,40.5134,23.82
2004-01-06,23.76,42.50,18.6019,46.5534,22.7548,23.1976,3.5625,15.2842,25.05,16.8880,...,20.7591,18.7034,20.0005,12.2067,20.298,8.6639,60.0315,43.7667,41.0732,23.82
2004-01-07,23.52,43.39,18.3235,46.3323,22.7224,22.9974,3.5150,15.0854,25.05,16.5838,...,20.6083,18.6057,20.0716,12.1498,20.298,8.5155,60.7012,44.5000,40.7899,23.82
2004-01-08,23.36,43.59,19.1678,43.9361,22.6808,22.9807,3.5783,15.3015,25.05,16.7997,...,20.5580,18.7247,20.1284,12.2330,20.298,8.5403,61.8752,44.6000,41.0057,23.82


In [193]:
univ_list = pd.read_csv("../Data/data_us/univ_h.csv")
univ_list['year'] = pd.to_datetime(univ_list['year'], format = "%Y")
univ_list.set_index('year', inplace= True)

close_prices_dict = {}
stocks_list_dict = {}
volatility_dict = {}


for year in univ_list.index.unique():
    stocks_list = univ_list.columns[univ_list.loc[year] == 1].tolist()
    # Filter stocks_list to include only columns present in close_prices
    stocks_list = [stock for stock in stocks_list if stock in close_prices.columns]
    print(f"Year: {year.year} - Number of stocks: {len(stocks_list)}")
    
    stocks_list_dict[year.year] = stocks_list
    close_prices_dict[year.year] = close_prices
    
    close_prices_dict[year.year] = close_prices_dict[year.year].loc[str(year.year)]
    
    # Add 5 days from previous year if present
    if year.year - 1 in close_prices_dict:
        close_prices_dict[year.year] = pd.concat([close_prices_dict[year.year - 1].tail(5), close_prices_dict[year.year]])
        
    close_prices_dict[year.year] = close_prices_dict[year.year][stocks_list]
    
    
display(close_prices_dict[2010].head(10))


Year: 2004 - Number of stocks: 499
Year: 2005 - Number of stocks: 499
Year: 2006 - Number of stocks: 499
Year: 2007 - Number of stocks: 498
Year: 2008 - Number of stocks: 498
Year: 2009 - Number of stocks: 498
Year: 2010 - Number of stocks: 498
Year: 2011 - Number of stocks: 498
Year: 2012 - Number of stocks: 499
Year: 2013 - Number of stocks: 499
Year: 2014 - Number of stocks: 498
Year: 2015 - Number of stocks: 499
Year: 2016 - Number of stocks: 501
Year: 2017 - Number of stocks: 502
Year: 2018 - Number of stocks: 502
Year: 2019 - Number of stocks: 502
Year: 2020 - Number of stocks: 501
Year: 2021 - Number of stocks: 501
Year: 2022 - Number of stocks: 501
Year: 2023 - Number of stocks: 500
Year: 2024 - Number of stocks: 500


,0202445Q,0203524D,0772031D,0848680D,0961514D,1086832D,1255459D,1280712D,1284849D,1431816D,...,X,XEL,XLNX,XOM,XRAY,XRX,XTO,YUM,ZBH,ZION
Date,,,,,,,,,,,,,,,,,,,,,
2009-12-24,72.50,14.8151,4.0188,3.73,35.9470,20.4732,52.17,23.60,61.9231,5.9835,...,50.6806,12.8872,19.7420,38.9939,30.8990,12.7559,46.7119,19.0382,52.4135,9.8934
2009-12-28,72.69,14.6217,4.0188,3.65,35.9731,20.2354,52.70,23.51,62.7238,5.9537,...,50.3508,12.7977,19.6408,39.2325,31.0567,12.5052,47.0193,18.9198,52.4743,9.7860
2009-12-29,72.68,14.7377,3.9536,3.81,35.9818,20.1799,52.49,23.41,62.8820,5.9140,...,48.7642,12.7918,19.4773,39.0962,31.0480,12.7164,46.8453,18.9521,52.3352,9.7784
2009-12-30,72.81,14.9891,3.9536,3.78,36.1122,20.6238,52.43,23.45,62.7633,5.9537,...,49.4773,12.7977,19.7342,39.0564,31.1531,12.8349,46.7359,18.9467,51.9699,9.8167
2009-12-31,72.35,14.8538,3.9256,3.66,35.6428,20.6872,52.22,23.34,62.2888,5.9636,...,49.1297,12.6605,19.5085,38.7270,30.8114,12.5385,46.2686,18.8176,51.4047,9.8397
2010-01-04,72.21,14.8344,3.9722,3.90,35.5733,21.9316,51.96,23.70,62.5953,6.0331,...,51.6165,12.5770,19.7576,39.2722,30.9516,12.7905,47.1039,18.8821,52.1960,10.2232
2010-01-05,72.15,14.6120,4.1587,4.13,35.1387,22.3358,51.57,23.74,62.1307,6.0628,...,51.5452,12.4278,19.5085,39.4256,30.5836,12.8053,47.4022,18.8176,53.8484,10.5836
2010-01-06,72.69,14.6797,4.2053,4.09,35.2343,22.7559,52.42,23.42,61.7451,5.9835,...,53.8359,12.4517,19.3761,39.7663,30.7851,12.6867,47.7303,18.6830,53.8310,11.5039
2010-01-07,71.88,14.5153,4.3638,3.97,34.8107,23.0016,51.25,23.38,60.4205,5.9140,...,54.2904,12.3980,19.1815,39.6414,31.1881,12.7460,47.7204,18.6777,55.0659,12.7924


In [ ]:
volatility_dict = {}
five_day_return_dict = {}
norm_five_day_return_dict = {}
industry_v_dict = {}
returns_minus_industry_dict = {}
industry_returns_dict = {}
beta_dict = {}
ranked_beta_dict = {}
rsquared_dict = {}
ranked_rsquared_dict = {}
ranked_norm_five_day_return_dict = {}
industry_ranked_v_dict = {}

for year in close_prices_dict.keys():
    if year == 2004:
        continue
    else:
        volatility_dict[year] = (np.log(close_prices_dict[year]).diff(1).fillna(0)).rolling(window=21).std()
        volatility_dict[year] = volatility_dict[year].applymap(lambda x: max(0.005, x))
        five_day_return_dict[year] = (np.log(close_prices_dict[year]).diff(5).fillna(0))
        norm_five_day_return_dict[year] = five_day_return_dict[year] / volatility_dict[year]
        returns_minus_industry_dict[year] = (np.log(close_prices_dict[year]).diff(1).fillna(0))
        
        ranks = norm_five_day_return_dict[year].rank(axis=1, method = 'first', ascending=False)
        N = ranks.shape[1]
        
        # Redefine the factor in terms of the rank
        ranked_norm_five_day_return_dict[year] = (N + 1 - 2 * ranks) / (N - 1)
        # display(ranked_norm_five_day_return_dict[year].head())

        industry_v_dict[year] = {}
        industry_returns_dict[year] = {}
        industry_ranked_v_dict[year] = {}
        
        for key, value in gics_industries.items():
            mask = tickers[tickers['industry'] == key].index
            mask = [ticker for ticker in mask if ticker in norm_five_day_return_dict[year].columns]
            
            industry_v_dict[year][key] = norm_five_day_return_dict[year].loc[:, mask].mean(axis=1)
            industry_returns_dict[year][key] = returns_minus_industry_dict[year].loc[:, mask].mean(axis=1)
            industry_ranked_v_dict[year][key] = ranked_norm_five_day_return_dict[year].loc[:, mask].mean(axis=1)
            

            # subtract the industry average from the stock return
            norm_five_day_return_dict[year].loc[:, mask] = norm_five_day_return_dict[year].loc[:, mask].sub(industry_v_dict[year][key], axis=0)
            
            ranked_norm_five_day_return_dict[year].loc[:, mask] = ranked_norm_five_day_return_dict[year].loc[:, mask].sub(industry_ranked_v_dict[year][key], axis=0)

            # Subtract the industry returns
            returns_minus_industry_dict[year].loc[:, mask] = returns_minus_industry_dict[year].loc[:, mask].sub(industry_returns_dict[year][key], axis=0)

        norm_five_day_return_dict[year] = norm_five_day_return_dict[year].loc[str(year)]
        ranked_norm_five_day_return_dict[year] = ranked_norm_five_day_return_dict[year].loc[str(year)]
        returns_minus_industry_dict[year] = returns_minus_industry_dict[year].loc[str(year)].shift(-1).dropna()
        
        beta_dict[year] = {}
        rsquared_dict[year] = {}
        ranked_beta_dict[year] = {}
        ranked_rsquared_dict[year] = {}
        
        for t in norm_five_day_return_dict[year].index[:-1]:  # Exclude the last period as it doesn't have t+1 returns
            # Dependent variable: returns at time t+1
            R_t1 = returns_minus_industry_dict[year].loc[t]
            
            # Independent variable: factors at time t
            v_t = norm_five_day_return_dict[year].loc[t]
            
            ranked_v_t = ranked_norm_five_day_return_dict[year].loc[t]
            
             # Calculate beta(t)
            beta_t = (R_t1 * v_t).sum() / (v_t * v_t).sum()
            beta_dict[year][t] = beta_t
            
            ranked_beta_t = (R_t1 * ranked_v_t).sum() / (ranked_v_t * ranked_v_t).sum()
            ranked_beta_dict[year][t] = ranked_beta_t
            
            
            # Calculate residuals
            epsilon_t = R_t1 - beta_t * v_t
            ranked_epsilon_t = R_t1 - ranked_beta_t * ranked_v_t
            
            # Calculate R^2(t)
            rsquared_t = 1 - (epsilon_t ** 2).sum() / (R_t1 ** 2).sum()
            rsquared_dict[year][t] = rsquared_t
            
            ranked_rsquared_t = 1 - (ranked_epsilon_t ** 2).sum() / (R_t1 ** 2).sum()
            ranked_rsquared_dict[year][t] = ranked_rsquared_t


In [195]:
summary_df = pd.DataFrame(columns=['Year', 'avg_beta', 't_stat', 'T'])

for year in beta_dict.keys():
    avg_beta = np.mean(list(beta_dict[year].values()))
    t_stat = np.sqrt(len(beta_dict[year])) * avg_beta / np.std(list(beta_dict[year].values()))
    T = len(beta_dict[year])
    
    summary_df.loc[len(summary_df)] = [str(year), avg_beta, t_stat, T]

summary_df.set_index('Year', inplace=True)
display(summary_df)

,avg_beta,t_stat,T
Year,,,
2005,-0.000056,-1.345597,251
2006,-0.000104,-2.492859,250
2007,-0.000081,-1.293516,250
2008,-0.000402,-1.441021,252
2009,-0.000142,-1.128730,251
2010,-0.000050,-1.054175,251
2011,-0.000179,-2.216054,251
2012,0.000042,1.050802,249
2013,-0.000049,-1.433107,251


In [196]:
ranked_summary_df = pd.DataFrame(columns=['Year', 'avg_beta', 't_stat', 'T'])

for year in ranked_beta_dict.keys():
    avg_beta = np.mean(list(ranked_beta_dict[year].values()))
    t_stat = np.sqrt(len(ranked_beta_dict[year])) * avg_beta / np.std(list(ranked_beta_dict[year].values()))
    T = len(ranked_beta_dict[year])
    
    ranked_summary_df.loc[len(ranked_summary_df)] = [str(year), avg_beta, t_stat, T]

ranked_summary_df.set_index('Year', inplace=True)
display(ranked_summary_df)

,avg_beta,t_stat,T
Year,,,
2005,-0.000224,-1.760081,251
2006,-0.000405,-3.139583,250
2007,-0.000227,-1.091790,250
2008,-0.000949,-1.445390,252
2009,0.000039,0.099731,251
2010,-0.000142,-1.080040,251
2011,-0.000320,-1.717146,251
2012,0.000098,0.734907,249
2013,-0.000194,-1.827771,251
